# Image Colourisation: GAN Comparison
Implementation of a **Conditional GAN** (U-Net + PatchGAN) for 256x256 images using the COCO dataset.

This notebook is now modular: you can run training, stop it, and run the evaluation block at any time to see the current progress.

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
from train import train_gan_loop
from utils import compare_colourisations
from dataset import LabColorDataset
from networks import Generator
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# --- Configuration ---
IMAGE_SIZE = 256
BATCH_SIZE = 8
NUM_EPOCHS = 50 # Set your target. You can stop and resume anytime.
MAX_SAMPLES = 1000 

transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor()
])

# Dataset Path Detection
base_path = "data/coco" if os.path.exists("data/coco/train") else "../data/coco"
if not os.path.exists(os.path.join(base_path, "train")):
    print("ERROR: Dataset not found. Please run 'python scripts/setup_coco.py' first.")
else:
    print(f"Dataset found at: {base_path}")

## 1. Training Blocks
Run these cells to start or resume training. You can interrupt the kernel at any time; progress is saved at the end of every epoch.

In [ ]:
# --- 1.1 Regression Training ---
netG_reg, _, hist_reg = train_gan_loop(
    base_path=base_path,
    transform=transform,
    device=device,
    mode='regression',
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    resume=True
)

In [ ]:
# --- 1.2 Classification Training ---
netG_cls, _, hist_cls = train_gan_loop(
    base_path=base_path,
    transform=transform,
    device=device,
    mode='classification',
    use_rebalancing=True,
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    num_epochs=NUM_EPOCHS,
    resume=True
)

## 2. Evaluation & Comparison Block
Run this block at any time to compare the current state of both models. It will load the latest available checkpoints from the `models/` folder.

In [ ]:
# --- 2.1 Load Models from Checkpoints ---
def load_model(mode, device, image_size=256):
    path = f'models/checkpoint_{mode}.pth'
    if not os.path.exists(path):
        print(f"No checkpoint found for {mode}. Train the model first.")
        return None, None
    
    model = Generator(image_size=image_size, use_classification=(mode=='classification')).to(device)
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['netG_state_dict'])
    model.eval()
    print(f"Loaded {mode} model from epoch {checkpoint['epoch']+1}")
    return model, checkpoint.get('history', None)

eval_netG_reg, eval_hist_reg = load_model('regression', device)
eval_netG_cls, eval_hist_cls = load_model('classification', device)

# --- 2.2 Plot Metrics (if available) ---
if eval_hist_reg and eval_hist_cls:
    plt.figure(figsize=(14, 4))
    plt.subplot(1, 2, 1)
    plt.plot(eval_hist_reg['loss_G'], label='Reg Loss')
    plt.plot(eval_hist_cls['loss_G'], label='Cls Loss')
    plt.title('Training Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(eval_hist_reg['psnr'], label='Reg PSNR')
    plt.plot(eval_hist_cls['psnr'], label='Cls PSNR')
    plt.title('Validation PSNR')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# --- 2.3 Side-by-Side Visual Comparison ---
if eval_netG_reg and eval_netG_cls:
    val_dataset = LabColorDataset(os.path.join(base_path, "val"), transform=transform)
    val_loader = DataLoader(val_dataset, batch_size=10, shuffle=True) # Shuffle to see different images
    
    print("Generating comparison for 10 random samples from validation set...")
    compare_colourisations(eval_netG_reg, eval_netG_cls, val_loader, val_dataset, device, num_samples=10)